# Week 11 — Neural Network Surrogate Training

Same architecture family as W4-W10: MLP with H ∈ {16, 32}, four regularisation variants (plain / dropout / weight-decay / ensemble), 5-fold CV across 8 configs.

Re-trains on the latest data including W10 portal returns (20/20/25/40/30/30/40/50 pts).

W10 new bests: F3 (-0.0264) and F5 (7663.60). W10 also empirically confirmed F1 and F4 are deterministic (exact-repeat noise tests matched W3 and W6 to 16 decimal places).


In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_11')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y


In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta


## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()


F1: 20 pts, 2D, baseline RMSE = 0.0016
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.0026 ← BEST
    H= 32  ensemble    CV RMSE = 0.0026
    H= 32  dropout     CV RMSE = 0.0027
    H= 32  wd          CV RMSE = 0.0027
    H= 32  plain       CV RMSE = 0.0027
    H= 16  ensemble    CV RMSE = 0.0033
    H= 16  wd          CV RMSE = 0.0044
    H= 16  plain       CV RMSE = 0.0047
  → best: H=16, dropout, RMSE=0.0026 (-60.7% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [0.017999999225139618, -0.003000000026077032]



F2: 20 pts, 2D, baseline RMSE = 0.2365
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.1804 ← BEST
    H= 32  dropout     CV RMSE = 0.1823
    H= 16  ensemble    CV RMSE = 0.2206
    H= 32  wd          CV RMSE = 0.2810
    H= 32  plain       CV RMSE = 0.2858
    H= 32  ensemble    CV RMSE = 0.2902
    H= 16  wd          CV RMSE = 0.3001
    H= 16  plain       CV RMSE = 0.3224
  → best: H=16, dropout, RMSE=0.1804 (+23.7% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [1.0110000371932983, 0.9449999928474426]



F3: 25 pts, 3D, baseline RMSE = 0.0724
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0695 ← BEST
    H= 16  ensemble    CV RMSE = 0.0731
    H= 32  wd          CV RMSE = 0.0798
    H= 32  plain       CV RMSE = 0.0802
    H= 16  wd          CV RMSE = 0.0834
    H= 16  plain       CV RMSE = 0.0850
    H= 32  dropout     CV RMSE = 0.0872
    H= 16  dropout     CV RMSE = 0.0878
  → best: H=32, ensemble, RMSE=0.0695 (+3.9% vs baseline) ✓ beats baseline


...

F6: 30 pts, 5D, baseline RMSE = 0.6651
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.3297 ← BEST
    H= 32  ensemble    CV RMSE = 0.3353
    H= 32  wd          CV RMSE = 0.3392
    H= 32  plain       CV RMSE = 0.3427
    H= 16  plain       CV RMSE = 0.3433
    H= 16  wd          CV RMSE = 0.3465
    H= 16  dropout     CV RMSE = 0.3779
    H= 32  dropout     CV RMSE = 0.4141
  → best: H=16, ensemble, RMSE=0.3297 (+50.4% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.2930000126361847, -0.375, -0.5270000100135803, -0.628000020980835, -1.340999960899353]



F7: 40 pts, 6D, baseline RMSE = 0.6586
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.3387 ← BEST
    H= 32  dropout     CV RMSE = 0.3495
    H= 16  dropout     CV RMSE = 0.3786
    H= 16  plain       CV RMSE = 0.3822
    H= 16  wd          CV RMSE = 0.3844
    H= 32  ensemble    CV RMSE = 0.4041
    H= 32  wd          CV RMSE = 0.4687
    H= 32  plain       CV RMSE = 0.4777
  → best: H=16, ensemble, RMSE=0.3387 (+48.6% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-1.1790000200271606, -2.941999912261963, 1.8619999885559082, 0.8410000205039978, -3.5239999294281006, -0.9110000133514404]



F8: 50 pts, 8D, baseline RMSE = 1.1777
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.3980 ← BEST
    H= 32  dropout     CV RMSE = 0.4191
    H= 32  ensemble    CV RMSE = 0.4232
    H= 16  ensemble    CV RMSE = 0.4319
    H= 32  wd          CV RMSE = 0.4323
    H= 32  plain       CV RMSE = 0.4364
    H= 16  wd          CV RMSE = 0.4876
    H= 16  plain       CV RMSE = 0.4948
  → best: H=16, dropout, RMSE=0.3980 (+66.2% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.5529999732971191, -0.21299999952316284, -0.7760000228881836, -0.16599999368190765, 0.052000001072883606, 0.01600000075995922, -0.47600001096725464, 0.11100000143051147]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — now especially useful since W10 confirmed F1 is deterministic (W3's 3.65e-7 is a real reproducible peak, not noise).


In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')


F1: 20 pts, 14 positive, 6 zero-or-negative (70% positive)


Sign classifier trained. LOO accuracy = 80.00%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')


 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   16  dropout         0.0026      0.0016    -60.7%       ✗
 2   16  dropout         0.1804      0.2365    +23.7%       ✓
 3   32  ensemble        0.0695      0.0724     +3.9%       ✓
 4   16  ensemble        4.3437      9.6964    +55.2%       ✓
 5   16  plain         317.0966   1858.2304    +82.9%       ✓
 6   16  ensemble        0.3297      0.6651    +50.4%       ✓
 7   16  ensemble        0.3387      0.6586    +48.6%       ✓
 8   16  dropout         0.3980      1.1777    +66.2%       ✓
